# Dedup API Performance Test - Approach Comparison

Cell 1: pandas UDF with intra-partition ThreadPoolExecutor (distributed across executors)
Cell 2: driver-side `toPandas()` + ThreadPoolExecutor (all requests fired from the driver)
Cell 3: comparison query

In [ ]:
# APPROACH 1: pandas UDF with intra-partition concurrency

import requests
import traceback
import time
import json
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from pyspark.sql import functions as F
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import StructType, StructField, DoubleType, StringType, IntegerType

# ---- CONFIG ----
MODEL_ENDPOINT = "https://pcis-model-service-staging.apps.k8s.uscis.dhs.gov/api/v1/model/dedup/predict"
RECORD_COUNT = 100
PARTITION_SIZE = 200
THREADS_PER_PARTITION = 16
SOURCE_TABLE = "pcis_data_science.small_static_test_set_regression_testing"
OUTPUT_TABLE_UDF = "pcis_data_science.dedup_perf_test_udf"

BEARER_TOKEN = dbutils.secrets.get(scope="pcis_dhs", key="dedup_bearer_token")
HEADERS = {"Authorization": f"Bearer {BEARER_TOKEN}"}

RETRY_STATUS_CODES = {502, 503, 504}
MAX_RETRIES = 3
BACKOFF_BASE = 0.5

effective_partitions = min(PARTITION_SIZE, max(1, RECORD_COUNT // 50))

print(f"=== APPROACH 1: UDF ===")
print(f"endpoint:              {MODEL_ENDPOINT}")
print(f"record_count:          {RECORD_COUNT:,}")
print(f"effective_partitions:  {effective_partitions}")
print(f"threads_per_partition: {THREADS_PER_PARTITION}")

source_df = (
    spark.table(SOURCE_TABLE)
    .withColumnRenamed("dob1", "dateOfBirth1")
    .withColumnRenamed("cob1", "countryOfBirth1")
    .withColumnRenamed("coc1", "countryOfCitizenship1")
    .withColumnRenamed("dob2", "dateOfBirth2")
    .withColumnRenamed("cob2", "countryOfBirth2")
    .withColumnRenamed("coc2", "countryOfCitizenship2")
)

source_count = source_df.count()
multiplier = max(1, int(RECORD_COUNT // source_count) + 1)

dedup_df = (
    source_df
    .withColumn("_rep", F.explode(F.array_repeat(F.lit(1), multiplier)))
    .drop("_rep")
    .limit(RECORD_COUNT)
    .repartition(effective_partitions)
)

schema = StructType([
    StructField("request", StringType()),
    StructField("response_status_code", StringType()),
    StructField("response_text", StringType()),
    StructField("score", DoubleType()),
    StructField("error", StringType()),
    StructField("package_version", StringType()),
    StructField("attempts", IntegerType()),
    StructField("latency_ms", DoubleType()),
])


def _call_one(session, payload, endpoint):
    response_status_code, response_text, score, package_version = None, None, None, None
    error = "None"
    attempt_count = 0
    start = time.perf_counter()

    for attempt in range(MAX_RETRIES + 1):
        attempt_count = attempt + 1
        try:
            resp = session.post(url=endpoint, json=payload, verify=False, timeout=30)
            response_status_code = resp.status_code
            response_text = resp.text

            if response_status_code in RETRY_STATUS_CODES and attempt < MAX_RETRIES:
                time.sleep(BACKOFF_BASE * (2 ** attempt))
                continue

            try:
                response_json = resp.json()
            except ValueError:
                response_json = None
            if response_json and "candidates" in response_json:
                package_version = response_json.get("packageVersion")
                try:
                    score = response_json["candidates"][0]["score"]
                except (IndexError, KeyError):
                    score = None
            break

        except (requests.ConnectionError, requests.Timeout) as e:
            if attempt < MAX_RETRIES:
                time.sleep(BACKOFF_BASE * (2 ** attempt))
                continue
            error = f"{type(e).__name__}: {e}"
            break
        except Exception:
            error = traceback.format_exc()
            break

    latency_ms = (time.perf_counter() - start) * 1000.0
    return {
        "request": json.dumps(payload),
        "response_status_code": str(response_status_code) if response_status_code is not None else "None",
        "response_text": response_text,
        "score": score,
        "error": error,
        "package_version": package_version,
        "attempts": attempt_count,
        "latency_ms": latency_ms,
    }


def _build_payload(row):
    return {
        "targetService": "dedup",
        "candidates": [{
            "candidateId": row["partySourceId2"], "firstName": row["firstName2"], "middleName": row["middleName2"],
            "lastName": row["lastName2"], "fullName": row["fullName2"], "dateOfBirth": row["dateOfBirth2"],
            "aNumber": row["aNumber2"], "countryOfBirth": row["countryOfBirth2"],
            "countryOfCitizenship": row["countryOfCitizenship2"], "ssn": row["ssn2"], "fin": row["fin2"],
            "motherName": row["motherName2"], "fatherName": row["fatherName2"],
        }],
        "declared": {
            "partySourceId": row["partySourceId1"], "firstName": row["firstName1"], "middleName": row["middleName1"],
            "lastName": row["lastName1"], "fullName": row["fullName1"], "dateOfBirth": row["dateOfBirth1"],
            "aNumber": row["aNumber1"], "countryOfBirth": row["countryOfBirth1"],
            "countryOfCitizenship": row["countryOfCitizenship1"], "ssn": row["ssn1"], "fin": row["fin1"],
            "motherName": row["motherName1"], "fatherName": row["fatherName1"],
        },
    }


@pandas_udf(schema)
def dedup_udf(
    partySourceId2: pd.Series, firstName2: pd.Series, middleName2: pd.Series, lastName2: pd.Series, fullName2: pd.Series,
    dateOfBirth2: pd.Series, aNumber2: pd.Series, countryOfBirth2: pd.Series, countryOfCitizenship2: pd.Series,
    ssn2: pd.Series, fin2: pd.Series, motherName2: pd.Series, fatherName2: pd.Series,
    partySourceId1: pd.Series, firstName1: pd.Series, middleName1: pd.Series, lastName1: pd.Series, fullName1: pd.Series,
    dateOfBirth1: pd.Series, aNumber1: pd.Series, countryOfBirth1: pd.Series, countryOfCitizenship1: pd.Series,
    ssn1: pd.Series, fin1: pd.Series, motherName1: pd.Series, fatherName1: pd.Series,
) -> pd.DataFrame:
    session = requests.Session()
    session.headers.update(HEADERS)

    df_in = pd.DataFrame({
        "partySourceId2": partySourceId2, "firstName2": firstName2, "middleName2": middleName2,
        "lastName2": lastName2, "fullName2": fullName2, "dateOfBirth2": dateOfBirth2, "aNumber2": aNumber2,
        "countryOfBirth2": countryOfBirth2, "countryOfCitizenship2": countryOfCitizenship2,
        "ssn2": ssn2, "fin2": fin2, "motherName2": motherName2, "fatherName2": fatherName2,
        "partySourceId1": partySourceId1, "firstName1": firstName1, "middleName1": middleName1,
        "lastName1": lastName1, "fullName1": fullName1, "dateOfBirth1": dateOfBirth1, "aNumber1": aNumber1,
        "countryOfBirth1": countryOfBirth1, "countryOfCitizenship1": countryOfCitizenship1,
        "ssn1": ssn1, "fin1": fin1, "motherName1": motherName1, "fatherName1": fatherName1,
    })
    payloads = [_build_payload(r) for _, r in df_in.iterrows()]

    with ThreadPoolExecutor(max_workers=THREADS_PER_PARTITION) as pool:
        results = list(pool.map(lambda p: _call_one(session, p, MODEL_ENDPOINT), payloads))

    session.close()
    return pd.DataFrame(results)


run_start = time.perf_counter()

initial_df = dedup_df.select(
    "*",
    dedup_udf(
        "partySourceId2", "firstName2", "middleName2", "lastName2", "fullName2",
        "dateOfBirth2", "aNumber2", "countryOfBirth2", "countryOfCitizenship2",
        "ssn2", "fin2", "motherName2", "fatherName2",
        "partySourceId1", "firstName1", "middleName1", "lastName1", "fullName1",
        "dateOfBirth1", "aNumber1", "countryOfBirth1", "countryOfCitizenship1",
        "ssn1", "fin1", "motherName1", "fatherName1",
    ).alias("dedup_response")
)

df_udf = (
    initial_df
    .withColumn("score", F.col("dedup_response.score"))
    .withColumn("response_status_code", F.col("dedup_response.response_status_code"))
    .withColumn("error", F.col("dedup_response.error"))
    .withColumn("request", F.col("dedup_response.request"))
    .withColumn("attempts", F.col("dedup_response.attempts"))
    .withColumn("latency_ms", F.col("dedup_response.latency_ms"))
    .withColumn("run_at", F.current_timestamp())
    .withColumn("endpoint", F.lit(MODEL_ENDPOINT))
    .withColumn("approach", F.lit("udf"))
    .withColumn("package_version", F.col("dedup_response.package_version"))
    .withColumn("response_text", F.col("dedup_response.response_text"))
    .drop("dedup_response")
)

df_udf.write.mode("overwrite").saveAsTable(OUTPUT_TABLE_UDF)

elapsed_udf = time.perf_counter() - run_start
print(f"\n[UDF] Wall clock: {elapsed_udf:.1f}s  ({RECORD_COUNT / elapsed_udf:.1f} req/s)")

In [ ]:
# APPROACH 2: driver-side toPandas + ThreadPoolExecutor (no UDF, no Spark distribution)

import requests
import traceback
import time
import json
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from pyspark.sql import functions as F

# ---- CONFIG ----
MODEL_ENDPOINT = "https://pcis-model-service-staging.apps.k8s.uscis.dhs.gov/api/v1/model/dedup/predict"
RECORD_COUNT = 100
DRIVER_THREADS = 64
SOURCE_TABLE = "pcis_data_science.small_static_test_set_regression_testing"
OUTPUT_TABLE_DRIVER = "pcis_data_science.dedup_perf_test_driver"

BEARER_TOKEN = dbutils.secrets.get(scope="pcis_dhs", key="dedup_bearer_token")
HEADERS = {"Authorization": f"Bearer {BEARER_TOKEN}"}

RETRY_STATUS_CODES = {502, 503, 504}
MAX_RETRIES = 3
BACKOFF_BASE = 0.5

print(f"=== APPROACH 2: driver toPandas + ThreadPool ===")
print(f"endpoint:        {MODEL_ENDPOINT}")
print(f"record_count:    {RECORD_COUNT:,}")
print(f"driver_threads:  {DRIVER_THREADS}")

source_df = (
    spark.table(SOURCE_TABLE)
    .withColumnRenamed("dob1", "dateOfBirth1")
    .withColumnRenamed("cob1", "countryOfBirth1")
    .withColumnRenamed("coc1", "countryOfCitizenship1")
    .withColumnRenamed("dob2", "dateOfBirth2")
    .withColumnRenamed("cob2", "countryOfBirth2")
    .withColumnRenamed("coc2", "countryOfCitizenship2")
)

source_count = source_df.count()
multiplier = max(1, int(RECORD_COUNT // source_count) + 1)

dedup_df = (
    source_df
    .withColumn("_rep", F.explode(F.array_repeat(F.lit(1), multiplier)))
    .drop("_rep")
    .limit(RECORD_COUNT)
)

pdf = dedup_df.toPandas()
print(f"pdf rows pulled to driver: {len(pdf):,}")


def _call_one_driver(session, payload):
    response_status_code, response_text, score, package_version = None, None, None, None
    error = "None"
    attempt_count = 0
    start = time.perf_counter()

    for attempt in range(MAX_RETRIES + 1):
        attempt_count = attempt + 1
        try:
            resp = session.post(url=MODEL_ENDPOINT, json=payload, verify=False, timeout=30)
            response_status_code = resp.status_code
            response_text = resp.text

            if response_status_code in RETRY_STATUS_CODES and attempt < MAX_RETRIES:
                time.sleep(BACKOFF_BASE * (2 ** attempt))
                continue

            try:
                response_json = resp.json()
            except ValueError:
                response_json = None
            if response_json and "candidates" in response_json:
                package_version = response_json.get("packageVersion")
                try:
                    score = response_json["candidates"][0]["score"]
                except (IndexError, KeyError):
                    score = None
            break

        except (requests.ConnectionError, requests.Timeout) as e:
            if attempt < MAX_RETRIES:
                time.sleep(BACKOFF_BASE * (2 ** attempt))
                continue
            error = f"{type(e).__name__}: {e}"
            break
        except Exception:
            error = traceback.format_exc()
            break

    latency_ms = (time.perf_counter() - start) * 1000.0
    return {
        "request": json.dumps(payload),
        "response_status_code": str(response_status_code) if response_status_code is not None else "None",
        "response_text": response_text,
        "score": score,
        "error": error,
        "package_version": package_version,
        "attempts": attempt_count,
        "latency_ms": latency_ms,
    }


def _build_payload_driver(row):
    return {
        "targetService": "dedup",
        "candidates": [{
            "candidateId": row["partySourceId2"], "firstName": row["firstName2"], "middleName": row["middleName2"],
            "lastName": row["lastName2"], "fullName": row["fullName2"], "dateOfBirth": row["dateOfBirth2"],
            "aNumber": row["aNumber2"], "countryOfBirth": row["countryOfBirth2"],
            "countryOfCitizenship": row["countryOfCitizenship2"], "ssn": row["ssn2"], "fin": row["fin2"],
            "motherName": row["motherName2"], "fatherName": row["fatherName2"],
        }],
        "declared": {
            "partySourceId": row["partySourceId1"], "firstName": row["firstName1"], "middleName": row["middleName1"],
            "lastName": row["lastName1"], "fullName": row["fullName1"], "dateOfBirth": row["dateOfBirth1"],
            "aNumber": row["aNumber1"], "countryOfBirth": row["countryOfBirth1"],
            "countryOfCitizenship": row["countryOfCitizenship1"], "ssn": row["ssn1"], "fin": row["fin1"],
            "motherName": row["motherName1"], "fatherName": row["fatherName1"],
        },
    }


run_start = time.perf_counter()

session = requests.Session()
session.headers.update(HEADERS)

payloads = [_build_payload_driver(r) for _, r in pdf.iterrows()]

with ThreadPoolExecutor(max_workers=DRIVER_THREADS) as pool:
    results = list(pool.map(lambda p: _call_one_driver(session, p), payloads))

session.close()

elapsed_driver = time.perf_counter() - run_start
print(f"\n[DRIVER] Wall clock: {elapsed_driver:.1f}s  ({RECORD_COUNT / elapsed_driver:.1f} req/s)")

results_df = pd.DataFrame(results)
results_df["run_at"] = pd.Timestamp.now()
results_df["endpoint"] = MODEL_ENDPOINT
results_df["approach"] = "driver"

spark.createDataFrame(results_df).write.mode("overwrite").saveAsTable(OUTPUT_TABLE_DRIVER)

## Compare results

In [ ]:
spark.sql("""
SELECT
  'udf' AS approach,
  COUNT(*) AS n,
  SUM(CASE WHEN response_status_code = '200' THEN 1 ELSE 0 END) AS success,
  SUM(CASE WHEN attempts > 1 THEN 1 ELSE 0 END) AS retried,
  ROUND(AVG(latency_ms), 1) AS avg_ms,
  ROUND(percentile_approx(latency_ms, 0.50), 1) AS p50_ms,
  ROUND(percentile_approx(latency_ms, 0.95), 1) AS p95_ms,
  ROUND(percentile_approx(latency_ms, 0.99), 1) AS p99_ms
FROM pcis_data_science.dedup_perf_test_udf

UNION ALL

SELECT
  'driver' AS approach,
  COUNT(*) AS n,
  SUM(CASE WHEN response_status_code = '200' THEN 1 ELSE 0 END) AS success,
  SUM(CASE WHEN attempts > 1 THEN 1 ELSE 0 END) AS retried,
  ROUND(AVG(latency_ms), 1) AS avg_ms,
  ROUND(percentile_approx(latency_ms, 0.50), 1) AS p50_ms,
  ROUND(percentile_approx(latency_ms, 0.95), 1) AS p95_ms,
  ROUND(percentile_approx(latency_ms, 0.99), 1) AS p99_ms
FROM pcis_data_science.dedup_perf_test_driver
""").display()